
# 08. Benchmarking, Keyboard Engine, Advanced Decoding, and Deployment

## Objectives
- Build production-style inference API
- Compare decoding strategies (beam, temperature, top-k, top-p)
- Simulate smartphone keyboard suggestion bar
- Prepare Streamlit deployment workflow


In [1]:

from pathlib import Path
import json
import pandas as pd
import torch

from utils.keyboard_engine import PredictiveKeyboardEngine
from utils.models import LSTM_LM, StackedLSTM_LM, BiLSTM_LM, GRU_LM, CNN_LSTM_LM, TransformerLM
from utils.vocabulary import Vocabulary

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS = PROJECT_ROOT / "outputs" / "results"

leaderboard_path = sorted(RESULTS.glob("leaderboard_*.csv"))[-1]
registry_path = sorted(RESULTS.glob("model_registry_*.json"))[-1]

leaderboard = pd.read_csv(leaderboard_path)
registry = json.loads(registry_path.read_text(encoding="utf-8"))

leaderboard.head(10)


/home/ahmad/AI/Github/40 AI-ML Projects for Beginners/Core Machine Learning and Data Science/Building a Predictive Keyboard Model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model,family,cross_entropy,perplexity,top1_accuracy,top3_accuracy,top5_accuracy,recall_at_3,recall_at_5,mrr,avg_latency_ms,throughput_examples_per_sec,memory_rss_mb_before,memory_rss_mb_after,memory_delta_mb,timed_batches,train_time_sec,rank_score
0,BiLSTM,neural,4.666769,1.063535e+02,0.181039,0.310565,0.384864,0.310565,0.384864,0.279847,0.985785,126675.711053,1842.953125,1842.953125,0.000000,24.0,7.164784,33.168714
1,LSTM,neural,4.681628,1.079457e+02,0.162810,0.300093,0.373741,0.300093,0.373741,0.269392,1.024631,121873.117889,1814.968750,1814.988281,0.019531,24.0,9.567106,31.976807
2,GRU,neural,4.767888,1.176704e+02,0.165334,0.298982,0.369374,0.298982,0.369374,0.265408,1.325289,94224.725519,1843.343750,1843.343750,0.000000,24.0,3.523280,31.053881
3,CNN_LSTM,neural,4.821506,1.241519e+02,0.169295,0.292963,0.358798,0.292963,0.358798,0.264275,1.215639,102723.770396,1961.960938,1961.960938,0.000000,24.0,4.675759,29.672173
4,StackedLSTM,neural,4.818019,1.237197e+02,0.155838,0.291630,0.357274,0.291630,0.357274,0.256875,1.694281,73703.825286,1834.406250,1834.406250,0.000000,24.0,6.307962,29.541463
5,Transformer,neural,4.817051,1.236001e+02,0.137124,0.275114,0.352017,0.275114,0.352017,0.241543,1.622672,76956.388915,2104.664062,2104.664062,0.000000,24.0,8.244779,29.021697
6,Bigram,ngram,6.013678,4.089848e+02,0.187109,0.325000,0.391797,0.325000,0.391797,0.278482,353.785309,361.801344,1293.410156,1293.417969,0.007812,5.0,NaN,18.730446
7,Unigram,ngram,5.325987,2.056111e+02,0.147266,0.208984,0.279297,0.208984,0.279297,0.217160,1719.308417,74.448539,1293.269531,1293.269531,0.000000,5.0,NaN,17.649134
8,Trigram,ngram,6.538057,6.909430e+02,0.160547,0.266016,0.319922,0.266016,0.319922,0.214741,855.200178,149.672560,1297.843750,1297.851562,0.007812,5.0,NaN,6.992188
9,MostFrequent,ngram,23.540335,1.672775e+10,0.148047,0.148047,0.191016,0.148047,0.191016,0.149013,18.039334,7095.605636,1293.195312,1293.195312,0.000000,5.0,NaN,-5.898438


In [2]:

MODEL_BUILDERS = {
    "LSTM": lambda c: LSTM_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), num_layers=1),
    "StackedLSTM": lambda c: StackedLSTM_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), num_layers=2),
    "BiLSTM": lambda c: BiLSTM_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=max(int(c["hidden_dim"])//2, 64), num_layers=2),
    "GRU": lambda c: GRU_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), num_layers=2),
    "CNN_LSTM": lambda c: CNN_LSTM_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), num_filters=max(int(c["embedding_dim"])//2, 64)),
    "Transformer": lambda c: TransformerLM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), nhead=int(c["transformer_heads"]), num_layers=int(c["transformer_layers"])),
}

best_model_name = str(leaderboard.iloc[0]["model"])
cfg = registry[best_model_name]
vocab_path = Path(str(cfg.get("vocab_path", PROJECT_ROOT / "outputs" / "vocab.json")))
vocab = Vocabulary.load(vocab_path)

model = MODEL_BUILDERS[best_model_name](cfg)
checkpoint = torch.load(Path(cfg["checkpoint_path"]), map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])

engine = PredictiveKeyboardEngine(model=model, vocabulary=vocab, context_length=int(cfg["context_len"]), device="cpu")
print("Loaded model:", best_model_name)


Loaded model: BiLSTM


In [3]:

prompt = "I would like to"

print("Top-3 suggestions:")
for row in engine.predict(prompt, top_k=3):
    print(row)

print("\nTop-5 suggestions:")
for row in engine.predict(prompt, top_k=5):
    print(row)

print("\nBeam suggestions:")
for row in engine.predict(prompt, top_k=5, strategy="beam"):
    print(row)


Top-3 suggestions:


{'token_id': 29, 'token': 'be', 'probability': 0.09594430774450302}
{'token_id': 4, 'token': 'the', 'probability': 0.0958094447851181}
{'token_id': 1, 'token': '<unk>', 'probability': 0.07035692781209946}

Top-5 suggestions:


{'token_id': 29, 'token': 'be', 'probability': 0.09594430774450302}
{'token_id': 4, 'token': 'the', 'probability': 0.0958094447851181}
{'token_id': 1, 'token': '<unk>', 'probability': 0.07035692781209946}
{'token_id': 28, 'token': 'me', 'probability': 0.058308545500040054}
{'token_id': 48, 'token': 'do', 'probability': 0.025845380499958992}

Beam suggestions:


{'token_id': 29, 'token': 'be', 'probability': 0.09594430029392242}
{'token_id': 4, 'token': 'the', 'probability': 0.0958094522356987}
{'token_id': 1, 'token': '<unk>', 'probability': 0.07035692036151886}


In [4]:

simulation = engine.simulate_keyboard_step("the meaning of", suggestion_count=3)
simulation


{'typed_text': 'the meaning of',
 'suggestions': [{'token_id': 4,
   'token': 'the',
   'probability': 0.212698295712471},
  {'token_id': 16, 'token': 'his', 'probability': 0.08231312036514282},
  {'token_id': 9, 'token': 'a', 'probability': 0.06559251993894577}],
 'best_completion': {'input': 'the meaning of',
  'completed_text': 'the meaning of the',
  'confidence': 0.212698295712471}}


## Deployment
Run local app:

```bash
uv run streamlit run app/streamlit_app.py
```

App exposes top-3/top-5 suggestions, probabilities, autocomplete, and strategy controls.
